In [1]:
!pip install -U langchain langchain-core==1.0 langchain-community langchain-experimental langchain_google_genai

  Using cached langchain-1.4.9-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_experimental-0.4.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached langchain_google_genai-4.4.3-py3-none-any.whl.metadata (2.7 kB)
INFO: pip is looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain-1.4.8-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.7-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.6-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.5-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.4-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.3-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain-1.4.2-py3-none-any.whl.metadata (6.2 kB)
INFO: pip is still looking at multiple versions of langchain to determine which version is c

### Document Loaders

In [2]:
## Text Loader

from langchain_community.document_loaders import TextLoader

loader=TextLoader('speech.txt')
loader

KeyboardInterrupt: 

In [ ]:
text_documents=loader.load()
text_documents

In [ ]:
!pip install pypdf
## Reading a PDf File
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader('attention.pdf')
docs=loader.load()
docs

In [ ]:
type(docs[0])

In [ ]:
## Web based loader
from langchain_community.document_loaders import WebBaseLoader
import bs4
loader=WebBaseLoader(web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
                     bs_kwargs=dict(parse_only=bs4.SoupStrainer(
                         class_=("post-title","post-content","post-header")
                     ))
                     )

In [ ]:
loader.load()

In [ ]:
!pip install wikipedia

from langchain_community.document_loaders import WikipediaLoader
docs = WikipediaLoader(query="Agentic AI", load_max_docs=2).load()
len(docs)
print(docs)

### RAG with LangChain

**Embedding Techniques**

Converting text into vectors


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.colab import userdata

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=userdata.get('gemini_key'))

embeddings

In [ ]:
text="This is a tutorial on Gemini embedding"
query_result=embeddings.embed_query(text)
query_result

In [ ]:
len(query_result)

In [ ]:
from langchain_community.document_loaders import TextLoader

loader=TextLoader('speech.txt')
docs=loader.load()
docs

In [ ]:
# to combine multiple docs

#from langchain_community.document_loaders import TextLoader

#loader1=TextLoader('speech.txt')

## Reading a PDf File
#from langchain_community.document_loaders import PyPDFLoader
#loader2=PyPDFLoader('Extra content.pdf')

#from langchain_community.document_loaders.merge import MergedDataLoader

#loader_all = MergedDataLoader(loaders=[loader1, loader2])
#docs = loader_all.load()

In [ ]:
# Experiment with Text splitters as well using the links in python.langchain.com

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
final_documents=text_splitter.split_documents(docs)
final_documents

In [ ]:
!pip install chromadb
## Vector Embedding And Vector StoreDB
from langchain_community.vectorstores import Chroma

db=Chroma.from_documents(final_documents,embeddings)
db



In [ ]:
### Retrieve the results from query vectorstore db
query="It will be all the easier for us to conduct ourselves as belligerents"
#query="""The primary inputs are sunlight, water, and carbon dioxide.
#It will be all the easier for us to conduct ourselves as belligerents"""

retrieved_results=db.similarity_search(query)
print(retrieved_results)

In [ ]:
## Retrieval Chain, Document chain
!pip install langchain-classic

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>
"""
)

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

In [ ]:
retriever=db.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)

In [ ]:
retrieval_chain

In [ ]:
## Get the response form the LLM
response=retrieval_chain.invoke({"input":"why is it regarded as an irresponsible government?"})
#response=retrieval_chain.invoke({"input":"Describe Photosynthesis"})
response['answer']

In [ ]:
response

{'input': 'Describe Photosynthesis',
 'context': [Document(metadata={'source': 'Extra content.pdf', 'page_label': '1', 'creationdate': '2025-10-25T18:35:54+05:30', 'page': 0, 'total_pages': 1, 'moddate': '2025-10-25T18:35:54+05:30', 'creator': 'Microsoft® Word 2019', 'producer': 'Microsoft® Word 2019', 'author': 'Surya Surya'}, page_content='byproduct.  \n• Summary: Photosynthesis is the process where plants use sunlight, water, and carbon \ndioxide to create their own food (glucose) and release oxygen.  \n• Inputs: The primary inputs are sunlight, water, and carbon dioxide, with chlorophyll acting as \nthe catalyst to capture light energy.  \n• Location: The process mainly happens in the leaves, specifically within organelles called \nchloroplasts.'),
  Document(metadata={'total_pages': 1, 'source': 'Extra content.pdf', 'moddate': '2025-10-25T18:35:54+05:30', 'producer': 'Microsoft® Word 2019', 'page_label': '1', 'page': 0, 'creator': 'Microsoft® Word 2019', 'author': 'Surya Surya', '